# 03 · Cuatro representaciones en un mismo espacio

EmbeddingGemma 2 puede leer texto **y** ver imágenes, y deja ambas cosas en el mismo espacio de 768 dimensiones. Eso permite cuatro formas de representar un video:

| representación | qué entra al modelo | cómo se obtiene |
|---|---|---|
| **Texto** | título + tags + descripción | un vector de texto (notebook 02) |
| **Imagen** | la miniatura | un vector de imagen |
| **Imagen + texto** (fusión temprana) | miniatura y texto juntos | el modelo los lee en una sola pasada |
| **Fusión tardía** | los dos vectores anteriores | $\operatorname{normalize}(x_\text{texto} + x_\text{imagen})$ |

La pregunta de este notebook: **¿mirar la miniatura agrega algo que el texto no tenga?**

In [ ]:
import sys
from pathlib import Path

# Los notebooks viven en notebooks/, pero los datos (data/) y el código (analysis/, recolect/)
# están en la raíz del repo. Así funcionan abriéndolos desde cualquiera de las dos carpetas.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
# True para sobrescribir las figuras que usa el informe (informe/figs/)
GUARDAR_FIGS = False
FIGS = ROOT / "informe" / "figs"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analysis.metricas import DATA, REPRESENTACIONES, cargar_videos, matriz
from analysis.graficos import COLORES, INK, MUTED, GRID, estilo, thumb, corto, plot3d, lollipop

estilo()

In [ ]:
df = cargar_videos()
metricas = pd.read_csv(DATA / "metricas.csv", index_col="modelo")
proy = pd.read_parquet(DATA / "proyecciones.parquet")
assert (proy["pid"] == df["pid"]).all()

GEMMA = {rep: REPRESENTACIONES[rep] for rep in ("text", "image", "imgtext", "fusion")}
X = {rep: matriz(df, rep) for rep in GEMMA}
print(f"{len(df):,} videos, cada uno con 4 vectores de {X['text'].shape[1]} dimensiones")

## 1. ¿Texto e imagen hablan el mismo idioma?

Que estén en el mismo espacio no garantiza que queden mezclados. Comparamos la similitud entre el texto y la miniatura **del mismo video** contra la de pares al azar. Si el modelo conecta bien las dos modalidades, cada video debería parecerse más a su propia miniatura que a la de otro.

In [ ]:
Xt, Xi = X["text"], X["image"]
otro = np.random.default_rng(0).permutation(len(df))     # empareja cada video con otro al azar

mismo_video = np.einsum("ij,ij->i", Xt, Xi)              # texto_i · imagen_i
azar = np.einsum("ij,ij->i", Xt, Xi[otro])               # texto_i · imagen_j
texto_texto = np.einsum("ij,ij->i", Xt, Xt[otro])        # texto_i · texto_j

fig, ax = plt.subplots(figsize=(11, 4.5))
todos = np.concatenate([azar, mismo_video, texto_texto])
bins = np.linspace(todos.min(), todos.max(), 100)
ax.hist(azar, bins, color="#b5b5af", alpha=0.7, label="texto vs miniatura, videos distintos")
ax.hist(mismo_video, bins, color="#eb6834", alpha=0.7, label="texto vs miniatura, mismo video")
ax.hist(texto_texto, bins, color="#2a78d6", alpha=0.45, label="texto vs texto, videos distintos")
ax.set_xlabel("similitud coseno", color=MUTED)
ax.set_title("¿Se reconoce cada video en su miniatura?", loc="left", fontweight="bold", color=INK)
ax.legend(frameon=False)
ax.yaxis.set_visible(False)
ax.tick_params(colors=MUTED, length=0)
for sp in ax.spines.values():
    sp.set_visible(False)
plt.show()

print(f"mediana texto–miniatura: mismo video {np.median(mismo_video):.3f} · al azar {np.median(azar):.3f}")
print(f"mediana texto–texto al azar: {np.median(texto_texto):.3f}")

Dos cosas para mirar:

1. **Naranjo contra gris:** cuánto más se parece un texto a su propia miniatura que a una cualquiera. Mientras más separadas las dos campanas, mejor conecta el modelo cada imagen con su texto.
2. **Azul contra gris:** las similitudes texto–imagen viven en otro rango que las texto–texto, incluso al azar. Es la llamada *brecha de modalidad*: imágenes y textos ocupan zonas distintas de la esfera. Por eso no conviene comparar valores absolutos entre modalidades, sino **rankings** (quién está más cerca de quién), que es lo que hacen el kNN y el zero-shot del notebook 04.

## 2. Los vecinos de un video en cada espacio

La forma más directa de ver la diferencia: un mismo video de consulta (con borde negro) y sus 5 vecinos más cercanos en cada representación.

In [ ]:
def vecinos(i, n=5, guardar=None):
    fig, axes = plt.subplots(len(GEMMA), n + 1, figsize=(3.1 * (n + 1), 2.6 * len(GEMMA)))
    for fila, (rep, nombre) in zip(axes, GEMMA.items()):
        sims = X[rep] @ X[rep][i]
        for j, (ax, k) in enumerate(zip(fila, np.argsort(-sims)[:n + 1])):   # el primero es la consulta
            ax.imshow(thumb(df.youtube_id[k]))
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(j == 0)
                sp.set_color(INK)
                sp.set_linewidth(3)
            encabezado = "CONSULTA" if j == 0 else f"{sims[k]:.2f}"
            ax.set_title(f"{encabezado}\n{corto(df.title[k], 28)}", fontsize=9, color=INK)
        fila[0].set_ylabel(nombre.replace("Gemma · ", ""), fontsize=13, fontweight="bold", color=INK)
    fig.tight_layout()
    if guardar:
        fig.savefig(FIGS / guardar, dpi=200, bbox_inches="tight")
    plt.show()

i = df.index[df.title.str.contains("rugby", case=False)][0]   # cambia la palabra para ver otros casos
vecinos(i, guardar="vecinos.png" if GUARDAR_FIGS else None)

Lo típico: en la fila de **imagen** aparecen videos que *se ven* parecidos (colores, composición, una cancha, una cara en primer plano) aunque sean de otro tema; en la de **texto**, videos del mismo tema aunque sus miniaturas no se parezcan. Las dos fusiones buscan un punto medio.

## 3. Las métricas de las seis representaciones

Las mismas tres métricas del notebook 02, ahora con todas las representaciones juntas. Todas se calculan sobre exactamente los mismos videos.

In [ ]:
metricas

In [ ]:
fig = lollipop(metricas,
               [("knn_acc_original", "kNN accuracy (espacio original)"),
                ("knn_acc_umap", "kNN accuracy (UMAP 3D)"),
                ("NMI_kmeans_umap", "NMI K-Means (UMAP 3D)")],
               modelos=list(COLORES))
if GUARDAR_FIGS:
    fig.savefig(FIGS / "metricas_multimodal.png", dpi=200, bbox_inches="tight")
plt.show()

Dos cosas llaman la atención:

1. **La miniatura sola queda casi a la par del texto.** Las etiquetas de YT-8M se asignaron a cada video sobre todo a partir de metadatos, lo que debería favorecer al texto; aun así, el vocabulario se restringió a entidades que se pueden reconocer *visualmente*, y la miniatura captura bastante de eso.
2. **La fusión tardía es la mejor en las tres métricas.** Texto e imagen se equivocan en videos distintos, y al sumar los dos vectores cada uno corrige parte de los errores del otro. La fusión temprana (imagen + texto en una sola pasada) también mejora al texto, pero menos.

## 4. Los mapas

UMAP 3D de cada representación de imagen, coloreado por categoría. Compáralos con el de texto del notebook 02.

In [ ]:
plot3d(proy, "image", "UMAP 3D · solo miniatura")

In [ ]:
plot3d(proy, "imgtext", "UMAP 3D · imagen + texto (fusión temprana)")

In [ ]:
plot3d(proy, "fusion", "UMAP 3D · fusión tardía")

## Para llevar

- Texto e imagen quedan en el mismo espacio, pero con una brecha: se comparan mejor por rankings que por valores absolutos.
- La miniatura sola ya ordena los videos casi tan bien como el texto, y sumar ambos vectores (fusión tardía) es lo mejor que medimos.
- Que la imagen sola ya ubique videos con sentido es lo que permite, en la web, buscar **con una foto cualquiera** y etiquetar frames de un video (notebook 04).